# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Read the dataset
df = pd.read_csv("tic-tac-toe.csv")

# 2. Inspect the dataset
print("First 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nClass distribution:")
print(df["class"].value_counts())

print("\nUnique values in each column:")
for column in df.columns:
    print(f"{column}: {df[column].unique()}")

# 3. Convert categorical values to numeric
# Board values:
# b = empty, o = Player O, x = Player X

mapping = {
    "b": 0,
    "o": 1,
    "x": 2
}

feature_columns = df.columns[:9]

for column in feature_columns:
    df[column] = df[column].map(mapping)

# Convert target to numeric
df["class"] = df["class"].map({
    False: 0,
    True: 1,
    "False": 0,
    "True": 1
})

# 4. Separate inputs and output
X = df[feature_columns]
y = df["class"]

# 5. Normalize the input data
scaler = StandardScaler()
X = scaler.fit_transform(X)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nNormalized input example:")
print(X[:5])

First 5 rows:


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True



Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   TL      958 non-null    str  
 1   TM      958 non-null    str  
 2   TR      958 non-null    str  
 3   ML      958 non-null    str  
 4   MM      958 non-null    str  
 5   MR      958 non-null    str  
 6   BL      958 non-null    str  
 7   BM      958 non-null    str  
 8   BR      958 non-null    str  
 9   class   958 non-null    bool 
dtypes: bool(1), str(9)
memory usage: 76.8 KB

Class distribution:
class
True     626
False    332
Name: count, dtype: int64

Unique values in each column:
TL: <ArrowStringArray>
['x', 'o', 'b']
Length: 3, dtype: str
TM: <ArrowStringArray>
['x', 'o', 'b']
Length: 3, dtype: str
TR: <ArrowStringArray>
['x', 'o', 'b']
Length: 3, dtype: str
ML: <ArrowStringArray>
['x', 'o', 'b']
Length: 3, dtype: str
MM: <ArrowStringArray>
['o', 'b', 'x']
Length: 3, dtype: str
MR:

## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [16]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# 1. Split training and test data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 2. Create Sequential model
model = Sequential([
    Dense(64, activation="relu", input_shape=(9,)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(2, activation="softmax")
])

# 3. Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 4. Fit
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

# 5. Evaluate
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# 6. Save
model.save("tic-tac-toe.keras")


Epoch 1/100


C:\Users\pc\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


39/39 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.6422 - loss: 0.6387 - val_accuracy: 0.6429 - val_loss: 0.6489
Epoch 2/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6683 - loss: 0.5935 - val_accuracy: 0.6494 - val_loss: 0.6378
Epoch 3/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7026 - loss: 0.5592 - val_accuracy: 0.6623 - val_loss: 0.6311
Epoch 4/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7451 - loss: 0.5245 - val_accuracy: 0.6883 - val_loss: 0.6173
Epoch 5/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7843 - loss: 0.4832 - val_accuracy: 0.7143 - val_loss: 0.6017
Epoch 6/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8186 - loss: 0.4394 - val_accuracy: 0.7143 - val_loss: 0.5805
Epoch 7/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8399 - loss: 0.3955 - val_accuracy: 0.7532 - val_loss: 0.5518
Epoch 8/100
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8660 - loss: 0.3534 - val_accuracy: 0.7727 - val_loss: 0.

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [20]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

# Load the saved model (corrected file extension and loading method)
# Use .keras extension for native Keras format 
model = load_model("tic-tac-toe.keras")

# Select a few random rows from the test dataset using NumPy random sampling
np.random.seed(42)  # Set seed for reproducibility
sample_indices = np.random.choice(len(X_test), size=5, replace=False)
sample = X_test[sample_indices]

# Keep the corresponding actual labels using the same indices
actual = y_test.iloc[sample_indices].values  # Changed from y_test.loc[sample.index].values

# Make predictions
probabilities = model.predict(sample, verbose=0)

# Softmax returns probabilities for both classes.
# Select the class with the highest probability.
predicted = np.argmax(probabilities, axis=1)

# Create a comparison table
results = pd.DataFrame({
    "Actual": actual,
    "Predicted": predicted,
    "Correct": actual == predicted
})

print(results)

# Display the prediction probabilities as well
print("\nPrediction probabilities:")
print(probabilities)

# Overall result for the selected samples
print(f"\nCorrect predictions: {results['Correct'].sum()}/{len(results)}")

   Actual  Predicted  Correct
0       1          1     True
1       1          1     True
2       1          1     True
3       1          1     True
4       1          1     True

Prediction probabilities:
[[1.49998639e-03 9.98499990e-01]
 [2.45562452e-03 9.97544348e-01]
 [1.08840474e-07 9.99999881e-01]
 [1.35020971e-01 8.64979029e-01]
 [4.21123281e-02 9.57887650e-01]]

Correct predictions: 5/5


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [21]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

improved_model = Sequential([
    Dense(64, activation="relu", input_shape=(9,)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

improved_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = improved_model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

loss, acc = improved_model.evaluate(X_test, y_test, verbose=0)

print(f"Improved Loss: {loss:.4f}")
print(f"Improved Accuracy: {acc:.4f}")

Epoch 1/100


C:\Users\pc\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


77/77 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6552 - loss: 0.6341 - val_accuracy: 0.6364 - val_loss: 0.6472
Epoch 2/100
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6634 - loss: 0.6046 - val_accuracy: 0.6364 - val_loss: 0.6302
Epoch 3/100
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6879 - loss: 0.5806 - val_accuracy: 0.6753 - val_loss: 0.6116
Epoch 4/100
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7206 - loss: 0.5458 - val_accuracy: 0.7013 - val_loss: 0.6006
Epoch 5/100
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7745 - loss: 0.5126 - val_accuracy: 0.7273 - val_loss: 0.5812
Epoch 6/100
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7729 - loss: 0.4937 - val_accuracy: 0.7532 - val_loss: 0.5581
Epoch 7/100
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7908 - loss: 0.4591 - val_accuracy: 0.7597 - val_loss: 0.5409
Epoch 8/100
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8039 - loss: 0.4441 - val_accuracy: 0.7662 - val_loss: 0.

**Which approach(es) did you find helpful to improve your model performance?**

In [22]:
print("""
The approaches that helped improve the model were:

1. Increasing the number of layers and neurons.
2. Increasing the number of training epochs.
3. Adjusting the Adam learning rate.

The additional training epochs and deeper network allowed the model
to learn more complex relationships between the Tic Tac Toe board
configurations and the outcome.
""")


The approaches that helped improve the model were:

1. Increasing the number of layers and neurons.
2. Increasing the number of training epochs.
3. Adjusting the Adam learning rate.

The additional training epochs and deeper network allowed the model
to learn more complex relationships between the Tic Tac Toe board
configurations and the outcome.

